In [21]:
import pandas as pd
import numpy as np
import nltk
import re
import zipfile

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
with zipfile.ZipFile('Resume.zip', 'r') as zip_ref:
    zip_ref.extractall('./')


In [10]:
df = pd.read_csv("Resume.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (2484, 4)


In [11]:
resume_column = 'Resume_str'

In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    # Replace non-alphabetic characters with spaces to ensure words are separated
    text = re.sub(r'[^a-zA-Z]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces and strip leading/trailing spaces

    stop_words = set(stopwords.words('english'))
    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [13]:
df['clean_resume'] = df[resume_column].apply(clean_text)

In [14]:
job_description = """
Looking for a Data Scientist with skills in
Python, Machine Learning, Data Analysis,
SQL, Statistics, Deep Learning and NLP.
"""
job_description = clean_text(job_description)

In [15]:
documents = [job_description] + df['clean_resume'].tolist()

In [16]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

In [17]:
similarity_scores = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
)


In [18]:
df['Similarity Score'] = similarity_scores[0]

In [19]:
ranked_candidates = df.sort_values(
    by='Similarity Score',
    ascending=False
)

print("\nTop Candidates\n")

print(
    ranked_candidates[
        [resume_column, 'Similarity Score']
    ].head(10)
)



Top Candidates

                                             Resume_str  Similarity Score
1762           ENGINEERING AND QUALITY TECHNICIAN   ...          0.196273
1218         Pavithra  Shetty           Summary     ...          0.178488
2153           CORPORATE BANKING ASSISTANT, INTERN  ...          0.167071
1339           DATA ANALYST       Professional Summa...          0.148462
1142           CONSULTANT       Summary    College g...          0.146074
2291           ONLINE LEARNING COORDINATOR - PROGRAM...          0.142364
1040           SALES COORDINATOR       Summary    Cu...          0.129673
331            MASTER DATA MANAGER           Experie...          0.127685
1091           SALES ENGINEER           Summary    I...          0.127151
194              PROGRAM MANAGER & DESIGNER         ...          0.124268


In [20]:
required_skills = [
    "python",
    "machine learning",
    "sql",
    "statistics",
    "nlp",
    "deep learning"
]

print("\nSkill Gap Analysis\n")

top_resume = ranked_candidates.iloc[0]['clean_resume']

missing_skills = []

for skill in required_skills:
    # Check if the skill (as a whole word) is present in the cleaned resume
    if f' {skill} ' not in f' {top_resume} ':
        missing_skills.append(skill)

print("Missing Skills:", missing_skills)


Skill Gap Analysis

Missing Skills: ['nlp', 'deep learning']
